In [6]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: C:\\py_temp\\추출 후 파일 저장\\T.riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: C:\\py_temp\\추출 후 파일 저장\\T.riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: C:\\py_temp\\추출 후 파일 저장\\T.riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text() #총 몇건인지 알려줌
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt)) 
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10) #최대 페이지 찾기 (한페이지에 10건이기에 수정사항임)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# ********여기부터 제일 중요함

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1		# 논문 번호 쓰려고 넘버해놓음

# **반복문 두개이기에 들여쓰기 잘해야 함

#여기부터 반복문 2개 (왜? 15건 뽑는다고 하면, 1페이지 일때 1~10건, 2페이지일 때 1~5건)
for a in range(1, collect_page_cnt + 1) :		# 페이지
    time.sleep(3)

    html_2 = driver.page_source 		#1페이지 전체 소스 다 가져오는 코드 / 들여쓰기 주의
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')	 #1페이지 일 때 10건 다 가져오라는 코드

    for b in content_2 :		# 첫번 째 li 태그 빼옴
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text() 	#li가 여러개일 때 제목 있는 애들은 이렇게 뽑음
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            print('2.논문제목:',title)		# 74번줄에서 데이터 뽑음
            title2.append(title)		# 텍스트파일에 제목추가
            f.write('\n' + '2.논문제목:' + title)	

            writer = b.find('span','writer').get_text()		#논문저자 찾기
            print('3.저자:',writer)
            writer2.append(writer)		# 텍스트파일에 저자추가
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()		#소속기관 찾기
            print('4.소속기관:' , org)
            org2.append(org)		# 기관 추가
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )			# close : 파일 저장
            if no > collect_cnt :		# 사용자가 요청한 건수
                break		# 요청대로 되면 멈춤

            no += 1
            print("\n")

    time.sleep(1) # 페이지 변경 전 1초 대기

    a += 1		# 여기까지는 숫자이기에 아래서 글자로 바꿈. 글자만 누를 수 있기에
    b = str(a)		#여기서 숫자로

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")


# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd		#117~123까지 표 만드는 기능 (pandas 사용) 

# 표 만들어줘. 참고로 데이터프레임은 **빈칸** 있으면 안됨. 0으로 채우거나 아무 글자 집어넣어야함.
# 다만! pd.DataFrame() 이건 빈칸 있어도 그대로 해달라는 뜻임. 
df = pd.DataFrame()		
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, engine='openpyxl')		#표이름. 표이름을 엑셀로 저장해달라는 뜻
#engine='openpyxl' 이건 안 써도 됨. 웬만하면 자동으로 되어있음.

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
In this paper, we will conduct in-depth research on the Prevention of hospital infection of respiratory infectious diseases, and in the future, we will conduct in-depth research on more hospitals to avoid and reduce hospital infection. It is hoped that the study will contribute to the avoidance and reduction of hospital infections of respiratory infectious diseases.xample analysis. According to the evaluation results and the actual specific situation of the hospital, preventive measures are put forward according to the hospital infection prevention standard level.Before the risk of nosocomial infection is serious, one pedestrian line and one motor vehicle line are set in the hospital. The underground access line to the treatment area is closed, which can reserve an entrance to the treatment area.When the risk of nosocomial infection is serious, the space is transformed. This can integrate the treatment area into a treatment area. A ground entran

## 코드 변형 (파일 저장 경로 고정)

In [7]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')


#Step 3. 수집된 데이터를 저장할 파일 이름 및 경로 자동 설정하기
import os
from datetime import datetime

# 1. 저장할 기본 폴더 경로 설정 (경로 마지막에 꼭 \\ 를 붙여주세요)
save_path = "C:\\py_temp\\추출 후 파일 저장\\"

# 해당 폴더가 컴퓨터에 없으면 자동으로 생성해 주는 안전장치
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. '검색어_현재시간' 형태로 파일명 자동 생성
now = datetime.now().strftime("%Y%m%d_%H%M") # 파일명 중복을 막기 위한 현재 시간(년월일_시분)

ft_name = f"{save_path}{query_txt}_{now}.txt"
fc_name = f"{save_path}{query_txt}_{now}.csv"
fx_name = f"{save_path}{query_txt}_{now}.xlsx"

print(f"👉 수집된 데이터는 [{save_path}] 폴더에 자동 저장됩니다.")


#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text() #총 몇건인지 알려줌
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt)) 
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10) #최대 페이지 찾기 (한페이지에 10건이기에 수정사항임)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# ********여기부터 제일 중요함

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1		# 논문 번호 쓰려고 넘버해놓음

# **반복문 두개이기에 들여쓰기 잘해야 함

#여기부터 반복문 2개 (왜? 15건 뽑는다고 하면, 1페이지 일때 1~10건, 2페이지일 때 1~5건)
for a in range(1, collect_page_cnt + 1) :		# 페이지
    time.sleep(3)

    html_2 = driver.page_source 		#1페이지 전체 소스 다 가져오는 코드 / 들여쓰기 주의
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')	 #1페이지 일 때 10건 다 가져오라는 코드

    for b in content_2 :		# 첫번 째 li 태그 빼옴
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text() 	#li가 여러개일 때 제목 있는 애들은 이렇게 뽑음
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            print('2.논문제목:',title)		# 74번줄에서 데이터 뽑음
            title2.append(title)		# 텍스트파일에 제목추가
            f.write('\n' + '2.논문제목:' + title)	

            writer = b.find('span','writer').get_text()		#논문저자 찾기
            print('3.저자:',writer)
            writer2.append(writer)		# 텍스트파일에 저자추가
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()		#소속기관 찾기
            print('4.소속기관:' , org)
            org2.append(org)		# 기관 추가
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )			# close : 파일 저장
            if no > collect_cnt :		# 사용자가 요청한 건수
                break		# 요청대로 되면 멈춤

            no += 1
            print("\n")

    time.sleep(1) # 페이지 변경 전 1초 대기

    a += 1		# 여기까지는 숫자이기에 아래서 글자로 바꿈. 글자만 누를 수 있기에
    b = str(a)		#여기서 숫자로

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")


# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd		#117~123까지 표 만드는 기능 (pandas 사용) 

# 표 만들어줘. 참고로 데이터프레임은 **빈칸** 있으면 안됨. 0으로 채우거나 아무 글자 집어넣어야함.
# 다만! pd.DataFrame() 이건 빈칸 있어도 그대로 해달라는 뜻임. 
df = pd.DataFrame()		
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, engine='openpyxl')		#표이름. 표이름을 엑셀로 저장해달라는 뜻
#engine='openpyxl' 이건 안 써도 됨. 웬만하면 자동으로 되어있음.

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
👉 수집된 데이터는 [C:\py_temp\추출 후 파일 저장\] 폴더에 자동 저장됩니다.
In this paper, we will conduct in-depth research on the Prevention of hospital infection of respiratory infectious diseases, and in the future, we will conduct in-depth research on more hospitals to avoid and reduce hospital infection. It is hoped that the study will contribute to the avoidance and reduction of hospital infections of respiratory infectious diseases.xample analysis. According to the evaluation results and the actual specific situation of the hospital, preventive measures are put forward according to the hospital infection prevention standard level.Before the risk of nosocomial infection is serious, one pedestrian line and one motor vehicle line are set in the hospital. The underground access line to the treatment area is closed, which can reserve an entrance to the treatment area.When the risk of nosocomial infection is serious, the space is transformed. This can integrate the tre